# Customer trajectories and offer-model inputs

Build an audit table, then a customer-by-decision training table with pre-decision history, four offer arms, and purchase/spend outcomes.

In [1]:
# Import standard-library utilities for paths and JSON, plus pandas for tabular analysis.
from pathlib import Path
import json
import pandas as pd

# Resolve the project root whether the notebook is opened from the project root or notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
TRANSCRIPT_PATH = PROCESSED_DIR / 'transcript_cleaned.jsonl'
PORTFOLIO_PATH = PROCESSED_DIR / 'portfolio_cleaned.jsonl'
PROFILE_PATH = PROCESSED_DIR / 'profile_cleaned.jsonl'

print(f'Project root: {PROJECT_ROOT}')
print(f'Transcript path: {TRANSCRIPT_PATH}')
print(f'Portfolio path: {PORTFOLIO_PATH}')
print(f'Profile path: {PROFILE_PATH}')

Project root: D:\SanJose\sem2\CMPE252
Transcript path: D:\SanJose\sem2\CMPE252\data\processed\transcript_cleaned.jsonl
Portfolio path: D:\SanJose\sem2\CMPE252\data\processed\portfolio_cleaned.jsonl
Profile path: D:\SanJose\sem2\CMPE252\data\processed\profile_cleaned.jsonl


In [2]:
# Read the line-delimited transcript and turn each JSON event into one flat record.
records = []
with TRANSCRIPT_PATH.open(encoding='utf-8') as stream:
    for line in stream:
        if not line.strip():
            continue
        # Parse either the cleaned flat record or the original nested record.
        event = json.loads(line)
        value = event.pop('value', {}) or {}
        event['offer_id'] = event.get('offer_id', value.get('offer_id', value.get('offer id')))
        event['amount'] = event.get('amount', value.get('amount'))
        event['reward'] = event.get('reward', value.get('reward'))
        records.append(event)

# Convert the records to a dataframe and order events within each customer timeline.
events = pd.DataFrame(records).drop(columns=['offer_type'], errors='ignore')
events = events.sort_values(['person', 'time'], kind='stable').reset_index(drop=True)

print(f'Rows loaded: {len(events):,}')
print(f'Columns: {list(events.columns)}')

Rows loaded: 306,137
Columns: ['person', 'event', 'time', 'amount', 'offer_id', 'reward']


In [3]:
# Read the offer catalog and join its metadata to transcript rows by offer_id.
portfolio_records = []
with PORTFOLIO_PATH.open(encoding='utf-8') as stream:
    for line in stream:
        if line.strip():
            portfolio_records.append(json.loads(line))

# Rename the catalog id to match the transcript key and disambiguate reward columns.
portfolio = pd.DataFrame(portfolio_records).rename(columns={
    'id': 'offer_id',
    'reward': 'portfolio_reward',
})
# A left join keeps transaction and other non-offer events in the dataframe.
# many_to_one verifies that each offer_id maps to only one portfolio definition.
events = events.merge(portfolio, on='offer_id', how='left', validate='many_to_one')

print(f'Offer events matched to portfolio metadata: {events["offer_type"].notna().sum():,}')
print(f'Columns after metadata join: {list(events.columns)}')

Offer events matched to portfolio metadata: 167,184
Columns after metadata join: ['person', 'event', 'time', 'amount', 'offer_id', 'reward', 'portfolio_reward', 'channels', 'difficulty', 'duration', 'offer_type']


In [4]:
# Build one row per customer-offer pair from the event-level transcript.
# This creates a compact lifecycle table for downstream analysis or modeling.
offer_events = events[events['event'].isin(['offer received', 'offer viewed', 'offer completed'])].copy()
# Transcript timestamps are hours; convert elapsed intervals to days below.
offer_events['event_time'] = offer_events['time'].astype(float)

# Keep the first occurrence of each lifecycle event for each customer-offer.
offer_lifecycle = (
    offer_events.pivot_table(
        index=['person', 'offer_id'],
        columns='event',
        values='event_time',
        aggfunc='min',
    )
    .rename(columns={
        'offer received': 'received_time',
        'offer viewed': 'viewed_time',
        'offer completed': 'completed_time',
    })
    .reset_index()
)

# Attach one copy of the static offer metadata to each customer-offer row.
offer_metadata = (
    offer_events[['offer_id', 'offer_type', 'difficulty', 'duration', 'portfolio_reward']]
    .drop_duplicates('offer_id')
)
offer_trajectories = offer_lifecycle.merge(offer_metadata, on='offer_id', how='left', validate='many_to_one')
# Convert lifecycle timestamps into boolean outcomes and elapsed-time features.
offer_trajectories['completed'] = offer_trajectories['completed_time'].notna()
offer_trajectories['viewed'] = offer_trajectories['viewed_time'].notna()
offer_trajectories['days_to_view'] = (offer_trajectories['viewed_time'] - offer_trajectories['received_time']) / 24
offer_trajectories['days_to_complete'] = (offer_trajectories['completed_time'] - offer_trajectories['received_time']) / 24

# Transaction amounts are assigned to completed offers in the next cell.
offer_trajectories['transaction_amount'] = 0.0
offer_trajectories = offer_trajectories.sort_values(['person', 'received_time', 'offer_id'], kind='stable').reset_index(drop=True)

print(f'Offer trajectories: {len(offer_trajectories):,} rows')
print(f'Completed offers: {offer_trajectories["completed"].sum():,}')
print(f'Viewed offers: {offer_trajectories["viewed"].sum():,}')
display(offer_trajectories.head())

Offer trajectories: 63,288 rows
Completed offers: 28,996
Viewed offers: 49,135


,person,offer_id,completed_time,received_time,viewed_time,offer_type,difficulty,duration,portfolio_reward,completed,viewed,days_to_view,days_to_complete,transaction_amount
0,0009655768c64bdeb2e877511632db8f,5a8bc65990b245e5a138643cd4eb9837,NaN,168.0,192.0,informational,0.0,3.0,0.0,False,True,1.0,NaN,0.0
1,0009655768c64bdeb2e877511632db8f,3f207df678b143eea3cee63160fa8bed,NaN,336.0,372.0,informational,0.0,4.0,0.0,False,True,1.5,NaN,0.0
2,0009655768c64bdeb2e877511632db8f,f19421c1d4aa40978ebb69ca19b0e20d,414.0,408.0,456.0,bogo,5.0,5.0,5.0,True,True,2.0,0.25,0.0
3,0009655768c64bdeb2e877511632db8f,fafdcd668e3743c1bb461111dcafc2a4,528.0,504.0,540.0,discount,10.0,10.0,2.0,True,True,1.5,1.00,0.0
4,0009655768c64bdeb2e877511632db8f,2906b810c7d4411798c6938adc9daaa5,576.0,576.0,NaN,discount,10.0,7.0,2.0,True,False,NaN,0.00,0.0


In [5]:
# Match transactions to completed offers one-to-one at the exact event time.
# events is sorted stably by customer and time, so row order preserves
# appearance order for events sharing the same timestamp.
events = events.reset_index(drop=True).copy()
events['_event_order'] = events.index

completed_offers = (
    events.loc[events['event'].eq('offer completed'),
                ['person', 'time', 'offer_id', '_event_order']]
    .sort_values('_event_order', kind='stable')
    .drop_duplicates(['person', 'offer_id'], keep='first')
)
completed_offers['_match_order'] = completed_offers.groupby(
    ['person', 'time'], sort=False
).cumcount()

transactions = (
    events.loc[events['event'].eq('transaction'),
                ['person', 'time', 'amount', '_event_order']]
    .dropna(subset=['amount'])
    .sort_values('_event_order', kind='stable')
)
transactions['_match_order'] = transactions.groupby(
    ['person', 'time'], sort=False
).cumcount()

In [6]:
# This left join preserves every completed offer. Extra transactions remain
# available below as unmatched transaction rows in the audit table.
matched_transactions = completed_offers.merge(
    transactions[['person', 'time', '_match_order', 'amount', '_event_order']]
    .rename(columns={'_event_order': 'transaction_event_order'}),
    on=['person', 'time', '_match_order'],
    how='left',
    validate='one_to_one',
)
matched_transactions['transaction_matched'] = matched_transactions['amount'].notna()
matched_transactions = matched_transactions.rename(columns={
    'time': 'transaction_time',
    'amount': 'transaction_amount',
    '_event_order': 'offer_event_order',
})
matched_transactions['transaction_amount'] = matched_transactions['transaction_amount'].fillna(0.0)
# A completion timestamp is not a transaction timestamp when no transaction matched.
matched_transactions.loc[~matched_transactions['transaction_matched'], 'transaction_time'] = float('nan')

In [7]:
# Add the assigned transaction fields to every offer, including offers that
# were never completed and therefore cannot have a matching transaction.
offer_match_key = matched_transactions.set_index(['person', 'offer_id'])
offer_transaction_amounts = offer_match_key['transaction_amount'].to_dict()
offer_transaction_flags = offer_match_key['transaction_matched'].to_dict()
offer_transaction_times = offer_match_key['transaction_time'].to_dict()
offer_transaction_orders = offer_match_key['transaction_event_order'].to_dict()
offer_trajectories['transaction_amount'] = [
    offer_transaction_amounts.get((person, offer_id), 0.0)
    for person, offer_id in zip(offer_trajectories['person'], offer_trajectories['offer_id'])
]
offer_trajectories['transaction_matched'] = [
    offer_transaction_flags.get((person, offer_id), False)
    for person, offer_id in zip(offer_trajectories['person'], offer_trajectories['offer_id'])
]

In [8]:
offer_rows = offer_trajectories.copy()
offer_rows['audit_record_type'] = 'offer'
offer_rows['transaction_time'] = [
    offer_transaction_times.get((person, offer_id), pd.NA)
    for person, offer_id in zip(offer_rows['person'], offer_rows['offer_id'])
]
offer_rows['transaction_event_order'] = [
    offer_transaction_orders.get((person, offer_id), pd.NA)
    for person, offer_id in zip(offer_rows['person'], offer_rows['offer_id'])
]
offer_rows['audit_status'] = 'offer_not_completed'
offer_rows.loc[offer_rows['completed'] & offer_rows['transaction_matched'], 'audit_status'] = 'offer_completed_with_transaction'
offer_rows.loc[offer_rows['completed'] & ~offer_rows['transaction_matched'], 'audit_status'] = 'offer_completed_without_transaction'

In [9]:
# Append every transaction that was not consumed by an offer match.
used_transaction_orders = set(
    matched_transactions.loc[matched_transactions['transaction_matched'], 'transaction_event_order']
    .astype(int)
)
unmatched_transactions = transactions.loc[~transactions['_event_order'].isin(used_transaction_orders)].copy()
transaction_rows = unmatched_transactions.rename(columns={
    'time': 'transaction_time',
    'amount': 'transaction_amount',
    '_event_order': 'transaction_event_order',
}).drop(columns=['_match_order'])
for column in offer_rows.columns:
    if column not in transaction_rows.columns:
        transaction_rows[column] = pd.NA
transaction_rows['audit_record_type'] = 'transaction'
transaction_rows['audit_status'] = 'unmatched_transaction'
transaction_rows['transaction_matched'] = False
transaction_rows['completed'] = False
transaction_rows['viewed'] = False

audit_table = pd.concat([offer_rows, transaction_rows], ignore_index=True, sort=False)

C:\Users\makhi\AppData\Local\Temp\ipykernel_31792\1684211434.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  audit_table = pd.concat([offer_rows, transaction_rows], ignore_index=True, sort=False)


In [10]:
# Join cleaned customer demographics to every audit row.
profile_records = []
with PROFILE_PATH.open(encoding='utf-8') as stream:
    for line in stream:
        if line.strip():
            profile_records.append(json.loads(line))
profile = pd.DataFrame(profile_records).rename(columns={'id': 'person'})
profile['became_member_on'] = pd.to_datetime(profile['became_member_on'], errors='coerce')
audit_table = audit_table.merge(profile, on='person', how='left', validate='many_to_one')
audit_table = audit_table.sort_values(['person', 'transaction_time', 'audit_record_type'], kind='stable').reset_index(drop=True)

print(f'Audit rows: {len(audit_table):,}')
print(f'Offer rows: {(audit_table["audit_record_type"] == "offer").sum():,}')
print(f'Unmatched transaction rows: {(audit_table["audit_status"] == "unmatched_transaction").sum():,}')
print(f'Rows with demographics: {audit_table["income"].notna().sum():,}')
display(audit_table.head())

Audit rows: 175,204
Offer rows: 63,288
Unmatched transaction rows: 111,916
Rows with demographics: 175,204


,person,offer_id,completed_time,received_time,viewed_time,offer_type,difficulty,duration,portfolio_reward,completed,...,transaction_amount,transaction_matched,audit_record_type,transaction_time,transaction_event_order,audit_status,gender,age,became_member_on,income
0,0009655768c64bdeb2e877511632db8f,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,...,22.16,False,transaction,228,2,unmatched_transaction,M,33.0,2017-04-21,72000.0
1,0009655768c64bdeb2e877511632db8f,f19421c1d4aa40978ebb69ca19b0e20d,414.0,408.0,456.0,bogo,5.0,5.0,5.0,True,...,8.57,True,offer,414.0,6.0,offer_completed_with_transaction,M,33.0,2017-04-21,72000.0
2,0009655768c64bdeb2e877511632db8f,fafdcd668e3743c1bb461111dcafc2a4,528.0,504.0,540.0,discount,10.0,10.0,2.0,True,...,14.11,True,offer,528.0,10.0,offer_completed_with_transaction,M,33.0,2017-04-21,72000.0
3,0009655768c64bdeb2e877511632db8f,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,...,13.56,False,transaction,552,13,unmatched_transaction,M,33.0,2017-04-21,72000.0
4,0009655768c64bdeb2e877511632db8f,2906b810c7d4411798c6938adc9daaa5,576.0,576.0,NaN,discount,10.0,7.0,2.0,True,...,10.27,True,offer,576.0,15.0,offer_completed_with_transaction,M,33.0,2017-04-21,72000.0


In [11]:
# Summarize the structure and completeness of the joined event dataframe.
# The shape is reported as (number_of_rows, number_of_columns).
print('Data format:')
print(f'Shape: {events.shape}')

# Show the pandas dtype assigned to each column, such as object, int64, or float64.
print('\nColumn data types:')
print(events.dtypes)

# Count non-missing values in each column. Counts below the total row count
# indicate fields that are only present for certain event types.
print('\nNon-null counts:')
print(events.notna().sum())

Data format:
Shape: (306137, 12)

Column data types:
person               object
event                object
time                  int64
amount              float64
offer_id             object
reward              float64
portfolio_reward    float64
channels             object
difficulty          float64
duration            float64
offer_type           object
_event_order          int64
dtype: object

Non-null counts:


person              306137
event               306137
time                306137
amount              306137
offer_id            306137
reward              306137
portfolio_reward    167184
channels            167184
difficulty          167184
duration            167184
offer_type          167184
_event_order        306137
dtype: int64


## Model-table design
Each row is one profile customer at an observed offer-delivery wave. Arms are 0 = no offer, 1 = BOGO, 2 = discount, 3 = informational. Treatment comes from receipt, never from transaction matching. Outcomes cover [decision_time, decision_time + 72 hours); all history uses time strictly less than decision_time.

The processed files retain informational offers. Cleaned receipts and the cleaned catalog supply exposure metadata; cleaned transactions, completions, and profiles supply history and outcomes. This notebook uses only processed inputs. All profile customers are assumed eligible at each wave. Windows with prior active offers, multiple new offers, unknown offer types, or incomplete follow-up are retained in `decision_audit` but excluded from training.

The earlier `audit_table` summarizes customer/offer-ID pairs. This section uses individual receipt and completion events, preserving repeated offer deliveries. Same-time transactions and completion events are paired one-to-one in appearance order. Matching is an attribution assumption, not evidence of causation.

No confirmed calendar start date is supplied: `membership_days` remains missing until configured; membership join date is encoded instead. These are modeling inputs, not a fitted or causally validated uplift model.

In [12]:
# All transcript times are hours; catalog offer durations are days.
import numpy as np

FOLLOWUP_HOURS = 72
HOURS_PER_DAY = 24
# Set only when the real calendar date corresponding to transcript time zero is known.
STUDY_START_DATE = None
TREATMENT_ARMS = {0: 'no_offer', 1: 'bogo', 2: 'discount', 3: 'informational'}
ARM_BY_TYPE = {name: arm for arm, name in TREATMENT_ARMS.items()}
# Conservatively require the entire outcome window to end by the last observed event.
OBSERVATION_END_HOUR = float(events['time'].max())

In [13]:
# Use the cleaned event table and catalog loaded above for every treatment arm.
# The cleaning notebook now retains informational offers and their receipt events.
assert {'bogo', 'discount', 'informational'}.issubset(set(portfolio['offer_type'])), (
    'Rerun data_cleaning.ipynb with informational offers retained before this notebook.'
)
receipts = events.loc[
    events['event'].eq('offer received'),
    ['person', 'time', 'offer_id', '_event_order'],
].rename(columns={'_event_order': 'receipt_order'}).copy()
# Join metadata once from the processed catalog. Repeated deliveries stay distinct.
receipts = receipts.merge(
    portfolio[['offer_id', 'offer_type', 'duration']],
    on='offer_id', how='left', validate='many_to_one',
)
assert receipts['duration'].notna().all(), 'Every receipt needs a known duration.'
assert receipts['offer_type'].eq('informational').any(), (
    'The processed transcript must retain informational receipt events.'
)
# Receipt times are hours; catalog durations are days.
receipts['expiry_time'] = receipts['time'] + receipts['duration'] * HOURS_PER_DAY
print('Exposure records by type:', receipts['offer_type'].value_counts().to_dict())

Exposure records by type: {'discount': 30543, 'bogo': 30499, 'informational': 15235}


In [14]:
# Build rows for every profile customer, not just customers who bought something.
# Customers with no purchases can then receive a genuine zero outcome.
decision_times = np.sort(receipts['time'].unique())
customer_roster = profile[['person']].drop_duplicates()
assert len(customer_roster) == len(profile), 'Profiles must have unique customer IDs.'
assert set(events['person']).issubset(set(customer_roster['person']))
decision_grid = customer_roster.merge(
    pd.DataFrame({'decision_time': decision_times}), how='cross'
)
decision_grid['window_end'] = decision_grid['decision_time'] + FOLLOWUP_HOURS

# A single new receipt defines the treatment arm. Multiple receipts are ambiguous.
receipt_summary = receipts.groupby(['person', 'time'], sort=False).agg(
    new_offer_count=('offer_id', 'size'),
    assigned_offer_id=('offer_id', 'first'),
    assigned_offer_type=('offer_type', 'first'),
).reset_index().rename(columns={'time': 'decision_time'})
decision_grid = decision_grid.merge(
    receipt_summary, on=['person', 'decision_time'], how='left', validate='one_to_one'
)
decision_grid['new_offer_count'] = decision_grid['new_offer_count'].fillna(0).astype(int)
decision_grid['treatment'] = decision_grid['assigned_offer_type'].map(ARM_BY_TYPE).astype('Int64')
decision_grid.loc[decision_grid['new_offer_count'].eq(0), 'treatment'] = 0
decision_grid.loc[decision_grid['new_offer_count'].gt(1), 'treatment'] = pd.NA

In [15]:
# Exclude carryover exposure in all arms. Even a previously completed offer is
# conservatively treated as active through its catalog expiry.
eligibility_parts = []
for decision_time in decision_times:
    block = decision_grid.loc[decision_grid['decision_time'].eq(decision_time)].copy()
    prior_active = receipts.loc[
        receipts['time'].lt(decision_time) & receipts['expiry_time'].ge(decision_time)
    ].groupby('person').size()
    later_receipts = receipts.loc[
        receipts['time'].gt(decision_time)
        & receipts['time'].lt(decision_time + FOLLOWUP_HOURS)
    ].groupby('person').size()
    block['prior_active_offer_count'] = block['person'].map(prior_active).fillna(0).astype(int)
    block['later_offer_count'] = block['person'].map(later_receipts).fillna(0).astype(int)
    eligibility_parts.append(block)
decision_audit = pd.concat(eligibility_parts, ignore_index=True)
decision_audit['exclusion_reason'] = np.select(
    [
        decision_audit['window_end'].gt(OBSERVATION_END_HOUR),
        decision_audit['new_offer_count'].gt(1),
        decision_audit['treatment'].isna(),
        decision_audit['prior_active_offer_count'].gt(0),
        decision_audit['later_offer_count'].gt(0),
    ],
    ['incomplete_followup', 'multiple_new_offers', 'unknown_offer_type',
     'prior_active_offer', 'additional_offer_during_followup'],
    default='eligible',
)
decision_audit['eligible'] = decision_audit['exclusion_reason'].eq('eligible')
model_base = decision_audit.loc[decision_audit['eligible']].copy()
model_base['treatment'] = model_base['treatment'].astype(int)
model_base['treatment_name'] = model_base['treatment'].map(TREATMENT_ARMS)
print(decision_audit['exclusion_reason'].value_counts())

exclusion_reason
prior_active_offer    57633
eligible              44367
Name: count, dtype: int64


In [16]:
# Demographics are customer attributes available before the current decision.
# Use the supplied cleaned values; upstream cleaning already imputed age/income.
model_base = model_base.merge(profile, on='person', how='left', validate='many_to_one')
model_base['membership_join_day'] = (
    model_base['became_member_on'] - pd.Timestamp('1970-01-01')
).dt.days
model_base['membership_days'] = np.nan
if STUDY_START_DATE is not None:
    calendar_decision = pd.Timestamp(STUDY_START_DATE) + pd.to_timedelta(
        model_base['decision_time'], unit='h'
    )
    model_base['membership_days'] = (
        calendar_decision - model_base['became_member_on']
    ).dt.total_seconds() / (24 * 60 * 60)
    assert model_base['membership_days'].dropna().ge(0).all()
# Do not substitute today's date: that would give incorrect historical tenure.

In [17]:
# Match individual completion events, including repeated offer occurrences.
# Pair the first transaction with the first completion at a customer/time, etc.
# Unlike the earlier customer/offer summary, do not collapse repeated completions.
history_transactions = events.loc[
    events['event'].eq('transaction'), ['person', 'time', 'amount', '_event_order']
].copy()
history_completions = events.loc[
    events['event'].eq('offer completed'), ['person', 'time', 'offer_id', '_event_order']
].copy()
for frame in (history_transactions, history_completions):
    frame.sort_values('_event_order', kind='stable', inplace=True)
    frame['pair_order'] = frame.groupby(['person', 'time'], sort=False).cumcount()
completion_pairs = history_completions.merge(
    history_transactions.rename(columns={'_event_order': 'transaction_event_order'}),
    on=['person', 'time', 'pair_order'], how='left', validate='one_to_one',
)
used_ids = completion_pairs['transaction_event_order'].dropna()
assert used_ids.is_unique, 'A transaction must not be assigned to two completions.'
history_transactions['unmatched'] = ~history_transactions['_event_order'].isin(used_ids)
assert history_transactions['amount'].notna().all(), 'Resolve missing amounts before training.'
assert np.isfinite(history_transactions['amount']).all()
assert history_transactions['amount'].ge(0).all()
# Matching at time t cannot affect features at t: history below uses strictly < t.

In [18]:
# Compute cumulative history independently at each decision wave.
# Strictly earlier events prevent same-time/future purchases from leaking into X.
history_parts = []
for decision_time in decision_times:
    block = model_base.loc[model_base['decision_time'].eq(decision_time)].copy()
    past = history_transactions.loc[history_transactions['time'].lt(decision_time)]
    summary = past.groupby('person').agg(
        prior_purchase_count=('amount', 'size'),
        prior_total_spend=('amount', 'sum'),
        last_purchase_time=('time', 'max'),
    )
    independent = past.loc[past['unmatched']].groupby('person').agg(
        prior_unmatched_purchase_count=('amount', 'size'),
        prior_unmatched_spend=('amount', 'sum'),
    )
    for column in ['prior_purchase_count', 'prior_total_spend']:
        block[column] = block['person'].map(summary[column]).fillna(0)
    for column in independent.columns:
        block[column] = block['person'].map(independent[column]).fillna(0)
    last_purchase = block['person'].map(summary['last_purchase_time'])
    block['has_prior_purchase'] = last_purchase.notna().astype(int)
    block['days_since_last_purchase'] = (decision_time - last_purchase) / HOURS_PER_DAY
    past_receipts = receipts.loc[receipts['time'].lt(decision_time)].groupby('person').size()
    past_completions = history_completions.loc[
        history_completions['time'].lt(decision_time)
    ].groupby('person').size()
    block['prior_offers_received'] = block['person'].map(past_receipts).fillna(0)
    block['prior_offers_completed'] = block['person'].map(past_completions).fillna(0)
    history_parts.append(block)
training_table = pd.concat(history_parts, ignore_index=True)

In [19]:
# Outcomes include ALL cleaned purchases, whether or not matched to an offer.
# Every eligible customer gets the same 72-hour follow-up and zero if no purchase.
outcome_parts = []
for decision_time in decision_times:
    block = training_table.loc[training_table['decision_time'].eq(decision_time)].copy()
    followup = history_transactions.loc[
        history_transactions['time'].ge(decision_time)
        & history_transactions['time'].lt(decision_time + FOLLOWUP_HOURS)
    ]
    outcomes = followup.groupby('person').agg(
        purchase_count=('amount', 'size'), spend=('amount', 'sum')
    )
    block['purchase_count'] = block['person'].map(outcomes['purchase_count']).fillna(0).astype(int)
    block['purchased'] = block['purchase_count'].gt(0).astype(int)
    block['spend'] = block['person'].map(outcomes['spend']).fillna(0.0)
    outcome_parts.append(block)
training_table = pd.concat(outcome_parts, ignore_index=True)
training_table = training_table.sort_values(['decision_time', 'person']).reset_index(drop=True)
# IDs and assigned offer metadata remain for inspection but are excluded from X.
display(training_table.head())

,person,decision_time,window_end,new_offer_count,assigned_offer_id,assigned_offer_type,treatment,prior_active_offer_count,later_offer_count,exclusion_reason,...,prior_total_spend,prior_unmatched_purchase_count,prior_unmatched_spend,has_prior_purchase,days_since_last_purchase,prior_offers_received,prior_offers_completed,purchase_count,purchased,spend
0,0009655768c64bdeb2e877511632db8f,0,72,0,NaN,NaN,0,0,0,eligible,...,0.0,0.0,0.0,0,NaN,0.0,0.0,0,0,0.00
1,00116118485d4dfda04fdbaba9a87b5c,0,72,0,NaN,NaN,0,0,0,eligible,...,0.0,0.0,0.0,0,NaN,0.0,0.0,0,0,0.00
2,0011e0d4e6b944f998e987f904e8c1e5,0,72,1,3f207df678b143eea3cee63160fa8bed,informational,3,0,0,eligible,...,0.0,0.0,0.0,0,NaN,0.0,0.0,0,0,0.00
3,0020c2b971eb4e9188eac86d93036a77,0,72,1,fafdcd668e3743c1bb461111dcafc2a4,discount,2,0,0,eligible,...,0.0,0.0,0.0,0,NaN,0.0,0.0,2,1,49.63
4,0020ccbbb6d84e358d3414a3ff76cffd,0,72,0,NaN,NaN,0,0,0,eligible,...,0.0,0.0,0.0,0,NaN,0.0,0.0,1,1,16.27


In [20]:
# Explicit feature allowlist excludes current treatment, completion, and outcomes.
numeric_features = [
    'age', 'income', 'decision_time', 'prior_purchase_count', 'prior_total_spend',
    'has_prior_purchase', 'days_since_last_purchase',
    'prior_unmatched_purchase_count', 'prior_unmatched_spend',
    'prior_offers_received', 'prior_offers_completed',
]
tenure_feature = 'membership_days' if STUDY_START_DATE is not None else 'membership_join_day'
numeric_features.append(tenure_feature)
X = training_table[numeric_features].astype(float).copy()
# Fixed sentinels and explicit missing flags need no fitting on future data.
for column in numeric_features:
    X[column + '_missing'] = X[column].isna().astype(int)
    X[column] = X[column].fillna(-1.0)
gender = training_table['gender'].fillna('Unknown')
gender = gender.where(gender.isin(['F', 'M', 'O']), 'Unknown')
gender = pd.Series(pd.Categorical(gender, categories=['F', 'M', 'O', 'Unknown']), index=X.index)
X = pd.concat([X, pd.get_dummies(gender, prefix='gender', dtype=int)], axis=1)

# Keep treatment and labels separate for an uplift estimator or arm-specific models.
treatment = training_table['treatment'].rename('treatment')
y_purchase = training_table['purchased'].rename('purchased')
y_spend = training_table['spend'].rename('spend')
row_keys = training_table[['person', 'decision_time']].copy()
print('X shape:', X.shape)
print('Arms:', TREATMENT_ARMS)

X shape: (44367, 28)
Arms: {0: 'no_offer', 1: 'bogo', 2: 'discount', 3: 'informational'}


In [21]:
# Hold out entire later waves to evaluate future behavior of this customer cohort.
# Customers may recur across splits; this is not a test of unseen-customer performance.
usable_waves = np.sort(training_table['decision_time'].unique())
assert len(usable_waves) >= 3, 'Need at least three eligible decision waves.'
validation_start, test_start = usable_waves[-2:]
split = pd.Series('train', index=training_table.index, name='split')
split.loc[training_table['decision_time'].eq(validation_start)] = 'validation'
split.loc[training_table['decision_time'].eq(test_start)] = 'test'
split_indices = {name: split.index[split.eq(name)] for name in ['train', 'validation', 'test']}
# Apply any fitted scaler, encoder, propensity model, or imputer only on train.
assert training_table.loc[split.eq('train'), 'window_end'].max() <= validation_start
assert training_table.loc[split.eq('validation'), 'window_end'].max() <= test_start
display(pd.crosstab(split, treatment).rename(columns=TREATMENT_ARMS))

treatment,no_offer,bogo,discount,informational
split,,,,
test,503,592,580,311
train,10079,11709,11728,5846
validation,777,945,887,410


In [22]:
# Check row identity, exposure rules, and feature/label alignment before fitting.
assert not training_table.duplicated(['person', 'decision_time']).any()
assert training_table['prior_active_offer_count'].eq(0).all()
assert training_table['later_offer_count'].eq(0).all()
assert training_table.loc[treatment.eq(0), 'new_offer_count'].eq(0).all()
assert training_table.loc[treatment.ne(0), 'new_offer_count'].eq(1).all()
assert training_table['window_end'].le(OBSERVATION_END_HOUR).all()
assert np.isfinite(X.to_numpy(dtype=float)).all()
assert X.index.equals(treatment.index) and X.index.equals(y_purchase.index)
assert X.index.equals(y_spend.index)
assert training_table['prior_unmatched_purchase_count'].le(training_table['prior_purchase_count']).all()
assert training_table['prior_unmatched_spend'].le(training_table['prior_total_spend'] + 1e-8).all()
assert set(treatment.unique()) == set(TREATMENT_ARMS), 'At least one arm has no eligible rows.'
assert not {'purchased', 'spend', 'completed_time', 'transaction_amount', 'audit_status', 'treatment'}.intersection(X.columns)

arm_summary = training_table.groupby('treatment_name').agg(
    rows=('person', 'size'), customers=('person', 'nunique'),
    nonpurchase_windows=('purchased', lambda s: int(s.eq(0).sum())),
    purchase_rate=('purchased', 'mean'), average_spend=('spend', 'mean'),
)
display(arm_summary)
print('Ready: training_table, X, treatment, y_purchase, y_spend, row_keys, split_indices')
print('Observed arm differences are descriptive; offer assignment may be confounded.')

,rows,customers,nonpurchase_windows,purchase_rate,average_spend
treatment_name,,,,,
bogo,13246,10098,5886,0.555639,10.778301
discount,13195,11259,6417,0.513679,10.137911
informational,6567,5474,3304,0.496878,8.871675
no_offer,11359,7850,7512,0.338674,4.606985


Ready: training_table, X, treatment, y_purchase, y_spend, row_keys, split_indices
Observed arm differences are descriptive; offer assignment may be confounded.


## Save model-ready data
Run these cells after the modeling cells above. They save `training_table.parquet`, `X.parquet`, and `metadata.json` to `data/processed/model_ready/`. Both tables include the same `row_id`; it is an alignment key, not a model feature. The training table includes the split, treatment, and outcomes.

Parquet requires `pyarrow` in the notebook's Python environment (`%pip install pyarrow` if needed). Rerunning the save cells refreshes these three generated files.

In [23]:
# Create export copies so saving does not change the in-memory modeling objects.
# Fail before writing if features or split labels are no longer aligned.
import pyarrow  # Required by the Parquet writer and reader below.

assert training_table.index.is_unique
assert X.index.equals(training_table.index)
assert split.index.equals(training_table.index)
assert split.isin(['train', 'validation', 'test']).all()
assert not training_table.duplicated(['person', 'decision_time']).any()

# A customer/time key survives reordering and is shared by both exported tables.
export_row_ids = (
    training_table['person'].astype(str) + ':' + training_table['decision_time'].astype(str)
)
assert export_row_ids.is_unique
training_export = training_table.copy()
training_export.insert(0, 'row_id', export_row_ids)
training_export['split'] = split
features_export = X.copy()
features_export.insert(0, 'row_id', export_row_ids)
training_export = training_export.reset_index(drop=True)
features_export = features_export.reset_index(drop=True)
# These object columns contain text. Nullable string dtype preserves missing
# values consistently instead of changing NaN to None during Parquet export.
for export_frame in (training_export, features_export):
    for column in export_frame.select_dtypes(include='object').columns:
        export_frame[column] = export_frame[column].astype('string')

In [24]:
# Keep generated modeling files together; unrelated processed data is untouched.
MODEL_READY_DIR = PROCESSED_DIR / 'model_ready'
MODEL_READY_DIR.mkdir(parents=True, exist_ok=True)
training_output_path = MODEL_READY_DIR / 'training_table.parquet'
features_output_path = MODEL_READY_DIR / 'X.parquet'
metadata_output_path = MODEL_READY_DIR / 'metadata.json'

# Parquet retains numeric and datetime types. Avoid saving the pandas index:
# row_id provides explicit alignment, while split is stored with the labels.
training_export.to_parquet(training_output_path, engine='pyarrow', index=False)
features_export.to_parquet(features_output_path, engine='pyarrow', index=False)
print(f'Saved {len(training_export):,} rows and {len(X.columns)} model features.')

Saved 44,367 rows and 28 model features.


In [25]:
# Record the schema and split boundaries so another notebook can load the data
# without guessing which columns are features, labels, or identifiers.
model_metadata = {
    'schema_version': 1,
    'files': {'training_table': training_output_path.name, 'features': features_output_path.name},
    'row_count': len(training_export),
    'row_id_column': 'row_id',
    'row_key_columns': ['person', 'decision_time'],
    'feature_columns': X.columns.tolist(),
    'feature_dtypes': {name: str(dtype) for name, dtype in X.dtypes.items()},
    'treatment_column': 'treatment',
    'treatment_arms': {str(arm): name for arm, name in TREATMENT_ARMS.items()},
    'outcome_columns': {'purchase': 'purchased', 'spend': 'spend'},
    'split_column': 'split',
    'split_counts': {name: int(split.eq(name).sum()) for name in split_indices},
    'split_decision_hours': {
        name: sorted(training_table.loc[split.eq(name), 'decision_time'].unique().tolist())
        for name in split_indices
    },
    'followup_hours': FOLLOWUP_HOURS,
    'observation_end_hour': OBSERVATION_END_HOUR,
    'study_start_date': None if STUDY_START_DATE is None else str(STUDY_START_DATE),
    'membership_feature': tenure_feature,
}

In [26]:
# Store preparation assumptions alongside the data for reproducible interpretation.
model_metadata['preparation'] = {
    'history_window': 'All events strictly before decision_time.',
    'outcome_window': '[decision_time, decision_time + followup_hours).',
    'transaction_matching': 'One-to-one completion/transaction pairing by customer, time, appearance order.',
    'exclusions': 'Prior active offer, multiple new offers, unknown arm, later receipt, incomplete follow-up.',
    'cohort': 'All profile customers assumed eligible at each decision wave.',
    'split_policy': 'Earlier waves train; penultimate wave validation; last wave test. Customers may recur.',
    'numeric_missing': 'Fixed -1 sentinel plus a missingness indicator.',
    'gender_encoding': 'One-hot F, M, O, Unknown; absent or unrecognized values become Unknown.',
    'demographics': 'Cleaned profiles; age/income already imputed upstream.',
    'causal_limit': 'Prepared inputs do not establish random treatment assignment or causal validity.',
}
model_metadata['sources'] = {
    'transactions_and_completions': str(TRANSCRIPT_PATH.relative_to(PROJECT_ROOT)),
    'demographics': str(PROFILE_PATH.relative_to(PROJECT_ROOT)),
    'exposure_receipts': str(TRANSCRIPT_PATH.relative_to(PROJECT_ROOT)),
    'exposure_catalog': str(PORTFOLIO_PATH.relative_to(PROJECT_ROOT)),
}
# JSON stores the feature order and arm labels in a human-readable form.
with metadata_output_path.open('w', encoding='utf-8') as stream:
    json.dump(model_metadata, stream, indent=2, allow_nan=False)
    stream.write('\n')

In [27]:
# Reload all three files and verify values, dtypes, and row alignment survived.
# This also demonstrates how the files can be read in the training notebook.
saved_training = pd.read_parquet(training_output_path, engine='pyarrow')
saved_features = pd.read_parquet(features_output_path, engine='pyarrow')
with metadata_output_path.open(encoding='utf-8') as stream:
    saved_metadata = json.load(stream)
pd.testing.assert_frame_equal(saved_training, training_export)
pd.testing.assert_frame_equal(saved_features, features_export)
assert saved_metadata == model_metadata
assert saved_training['row_id'].equals(saved_features['row_id'])

# Select only metadata's feature columns when fitting: row_id is not a feature.
assert saved_features.columns.tolist() == ['row_id'] + saved_metadata['feature_columns']
assert saved_training['split'].value_counts().to_dict() == saved_metadata['split_counts']
for output_path in (training_output_path, features_output_path, metadata_output_path):
    print(f'Verified: {output_path} ({output_path.stat().st_size:,} bytes)')

Verified: D:\SanJose\sem2\CMPE252\data\processed\model_ready\training_table.parquet (2,743,721 bytes)
Verified: D:\SanJose\sem2\CMPE252\data\processed\model_ready\X.parquet (1,889,228 bytes)
Verified: D:\SanJose\sem2\CMPE252\data\processed\model_ready\metadata.json (4,121 bytes)
